# 33 — Búsqueda de artefactos de FORMATO (no de valor)

**Idea propia.** La duración funcionó porque la señal no estaba en el valor ("60 días") sino en el **formato** ("60 días + unos segundos"). Acá se buscan artefactos parecidos en **todos** los campos crudos del JSON: cómo viene escrito el dato, no qué dice.

**Plan:**
1. **Zona de duda:** OOF de la entrega actual (variante a exacta, Fase T). Filas con P(new) entre 0,2 y 0,8, más los errores. Ahí está el margen para mejorar.
2. **Features de formato** de cada campo:
   - **genéricas**, para cada campo y subcampo (dicts recursivos; en las listas, el primer elemento): estado (clave ausente / null / texto vacío / lista vacía / dict vacío / valor), largo de los textos, cantidad de claves de los dicts, largo de las listas, decimales y centavos de los números con coma;
   - **específicas:**
     - título original: mayúsculas, espacios dobles, signos, caracteres raros;
     - `permalink` contra el título actual;
     - precio: centavos, terminaciones;
     - fechas: segundos y milisegundos;
     - fotos: formato del id, mes de subida, ids repetidos o de otra publicación, orden;
     - orden de tags y de métodos de pago;
     - formato de `descriptions`;
     - precio entero/decimal en variaciones.
3. **Ranking dentro de la zona de duda:** cada feature sola, con un clasificador de un solo paso (grupo → clase mayoritaria) **ajustado con 4 folds y evaluado en el quinto**, rotando. Se reporta:
   - accuracy sola y ganancia sobre "predecir la clase mayoritaria";
   - diferencia de % `new` entre el grupo más `new` y el más `used` (elegidos con 4 folds, medidos en el quinto).
4. **Las 3-5 mejores** (una por campo) se suman al HistGB y al promedio final de la entrega, con comparación pareada fold por fold (criterio de siempre).

**Material:**
- `Tratamiento-de-variables-categóricas.pdf`:
  - datos faltantes, slides 30-67, en particular **NMAR** (slide 56: que falte depende del propio dato). Distinguir null / vacío / ausente es tratar el faltante como información;
  - *feature engineering*, slides 68-78 ("generar otra variable que sintetice…"; slide 78, cautela: saber con qué fin se genera).
- Accuracy y comparación de modelos: `Repaso de Metricas.pdf`.
- **No está en el material:** buscar artefactos de formato de manera sistemática, y el ranking univariado *out-of-fold* restringido a la zona de duda.

**Por qué no es leakage:** cada feature sale del JSON **de la propia fila**, sin `condition`, y todos los campos están en el test. Son determinísticas (no aprenden nada de los datos), así que van afuera del `Pipeline`, como `agregar_features_contenido`. La zona de duda solo se usa para **elegir** qué probar. La medición final es la CV de siempre sobre las 70.000 filas: el modelo se entrena con el train de cada fold y se evalúa en la validación. Se verifica además que cada artefacto elegido aparezca en el test con la misma frecuencia.

**Ya probado antes:** segundos y milisegundos de las fechas (Fase M), redondez del precio (Fase B, marginal), URL de la miniatura (Fase G), conteos y flags de nulos de los campos (Fase K). Se incluyen igual para que el barrido sea completo, marcados en la tabla.

In [1]:
import sys
sys.path.append("..")
import os, re, json, time, warnings, unicodedata
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)
SEED = 42
carpeta_lab = ".cache_laboratorio"
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1


def acc_por_fold(prob, filas=None):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= 0.5).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


# Columna de texto EXACTA de la variante a (Fase T), igual que en la 31: para el fold externo k,
# logística = PrediccionOOF en el train del fold; BETO = sin_j_k en las filas del fold j, sin_k en la validación.
logistica_fold = {k: np.load(os.path.join(carpeta_lab, "textos_31", f"logistica_fold{k}.npy")) for k in range(1, 6)}
carpeta_bert = os.path.join(carpeta_lab, "bert")
bert_global = np.full(n, np.nan)
for k in range(1, 6):
    d = np.load(os.path.join(carpeta_bert, f"sin_{k}.npz"))
    es_train = d["filas"] < n
    bert_global[d["filas"][es_train]] = d["prob"][es_train]
bert_pares = {}
for j in range(1, 6):
    for k in range(j + 1, 6):
        d = np.load(os.path.join(carpeta_bert, f"sin_{j}_{k}.npz"))
        p = np.full(n, np.nan)
        p[d["filas"]] = d["prob"]
        bert_pares[(j, k)] = bert_pares[(k, j)] = p


def columna_a(k):
    beto = bert_global.copy()
    for j in range(1, 6):
        if j != k:
            beto[folds == j] = bert_pares[(j, k)][folds == j]
    return (logistica_fold[k] + beto) / 2


# OOF de la entrega (variante a exacta, Fase T)
oof_t = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_combinado.csv"))
assert (oof_t["fold"].to_numpy() == folds).all()
ref_hgb = oof_t["(a) promedio simple"].to_numpy()
ref_final = 0.5 * oof_t["catboost (a) promedio simple"].to_numpy() + 0.5 * ref_hgb
pred_final = (ref_final >= 0.5).astype(int)

zona = ((ref_final >= 0.2) & (ref_final <= 0.8)) | (pred_final != y_np)
print(f"Entrega — HistGB {acc_por_fold(ref_hgb).mean():.4f} | final {acc_por_fold(ref_final).mean():.4f}")
print(f"Zona de duda + errores: {zona.sum()} filas ({zona.mean():.1%}); "
      f"de ellas, errores: {(pred_final != y_np)[zona].sum()} | % new en la zona: {y_np[zona].mean():.1%} | "
      f"accuracy de la entrega en la zona: {(pred_final == y_np)[zona].mean():.4f}")
print(f"Errores totales: {(pred_final != y_np).sum()} — dentro de 0,2-0,8: {((pred_final != y_np) & (ref_final >= 0.2) & (ref_final <= 0.8)).sum()}")

Entrega — HistGB 0.9425 | final 0.9435
Zona de duda + errores: 8527 filas (12.2%); de ellas, errores: 3953 | % new en la zona: 50.1% | accuracy de la entrega en la zona: 0.5364
Errores totales: 3953 — dentro de 0,2-0,8: 2295


## 1. Features de formato

**Qué hace `formato_fila`:** recibe una fila del JSON **sin parsear con pandas** (con `json.loads`, que conserva si un número vino como entero o con coma, y si un campo es `null`, `""` o `[]`) y devuelve un diccionario de features. Primero las genéricas (recorre todos los campos), después las específicas.

**Qué entra / qué sale:** entran los 100.000 JSON crudos (train + test). Sale una tabla de 100.000 filas por ~300 features. El test se usa solo para chequear que cada artefacto también aparezca ahí con la misma frecuencia.

In [2]:
def leer_crudo(ruta):
    with open(ruta) as f:
        return [json.loads(linea) for linea in f]


filas_train = leer_crudo("../data/train_data.jsonlines")
filas_test = leer_crudo("../data/test_data.jsonlines")


def estado(v, presente=True):
    if not presente:
        return "ausente"
    if v is None:
        return "null"
    if isinstance(v, bool):
        return f"bool_{v}"
    if isinstance(v, str):
        return "texto_vacio" if v == "" else "texto"
    if isinstance(v, list):
        return "lista_vacia" if not v else "lista"
    if isinstance(v, dict):
        return "dict_vacio" if not v else "dict"
    return type(v).__name__                      # int / float


def rutas(v, ruta, salida):
    # Todas las rutas de campos: dicts recursivos; en las listas, el primer elemento ("[0]")
    salida.add(ruta)
    if isinstance(v, dict):
        for k, x in v.items():
            rutas(x, f"{ruta}.{k}" if ruta else k, salida)
    elif isinstance(v, list) and v:
        rutas(v[0], f"{ruta}[0]", salida)


TODAS_LAS_RUTAS = set()
for fila in filas_train + filas_test:
    rutas(fila, "", TODAS_LAS_RUTAS)
TODAS_LAS_RUTAS = sorted(r for r in TODAS_LAS_RUTAS if r and r != "condition")


def buscar(fila, ruta):
    # Devuelve (presente, valor) siguiendo la ruta; "[0]" = primer elemento de la lista
    v = fila
    for parte in re.findall(r"[^.\[\]]+|\[0\]", ruta):
        if parte == "[0]":
            if not isinstance(v, list) or not v:
                return False, None
            v = v[0]
        else:
            if not isinstance(v, dict) or parte not in v:
                return False, None
            v = v[parte]
    return True, v


def decimales(x):
    texto = repr(float(x))
    return 0 if texto.endswith(".0") else len(texto.split(".")[1]) if "." in texto and "e" not in texto else -1


def genericas(fila):
    f = {}
    for ruta in TODAS_LAS_RUTAS:
        presente, v = buscar(fila, ruta)
        f[f"estado|{ruta}"] = estado(v, presente)
        if isinstance(v, str) and v:
            f[f"largo|{ruta}"] = len(v)
        elif isinstance(v, dict):
            f[f"n_claves|{ruta}"] = len(v)
        elif isinstance(v, list):
            f[f"n|{ruta}"] = len(v)
        elif isinstance(v, float):
            f[f"decimales|{ruta}"] = decimales(v)
            f[f"centavos|{ruta}"] = int(round(v * 100)) % 100
    return f


def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "-", texto).strip("-")


def texto_formato(prefijo, t):
    letras = [c for c in t if c.isalpha()]
    palabras = t.split()
    return {
        f"{prefijo}_largo": len(t), f"{prefijo}_n_palabras": len(palabras),
        f"{prefijo}_pct_mayusculas": np.mean([c.isupper() for c in letras]) if letras else np.nan,
        f"{prefijo}_todo_mayusculas": int(bool(letras) and all(c.isupper() for c in letras)),
        f"{prefijo}_pct_palabras_capitalizadas": np.mean([p[:1].isupper() for p in palabras]) if palabras else np.nan,
        f"{prefijo}_espacios_dobles": t.count("  "), f"{prefijo}_espacio_en_borde": int(t != t.strip()),
        f"{prefijo}_n_signos": sum(not c.isalnum() and not c.isspace() for c in t),
        f"{prefijo}_n_digitos": sum(c.isdigit() for c in t),
        f"{prefijo}_no_ascii": sum(ord(c) > 127 for c in t), f"{prefijo}_raros": sum(ord(c) > 255 for c in t),
        f"{prefijo}_exclamacion": t.count("!"), f"{prefijo}_asterisco": t.count("*"),
        f"{prefijo}_barra": t.count("/") + t.count("|"), f"{prefijo}_mas": t.count("+"),
        f"{prefijo}_puntos_seguidos": t.count(".."), f"{prefijo}_parentesis": t.count("("),
        f"{prefijo}_comillas": t.count('"') + t.count("'"), f"{prefijo}_guion": t.count("-"),
    }


def iso_a_ms(texto):
    return int(pd.Timestamp(texto).value // 1_000_000)


PATRON_FOTO_NUEVA = re.compile(r"^(\d+)-MLA(\d+)_(\d{2})(\d{4})$")
PATRON_FOTO_VIEJA = re.compile(r"^(\d+)-MLA(\d+)_(\d+)$")


def especificas(fila):
    f = {}
    id_item = fila["id"][3:]
    titulo = fila["title"]
    f.update(texto_formato("titulo", titulo))
    if isinstance(fila.get("warranty"), str):
        f.update({k: v for k, v in texto_formato("garantia", fila["warranty"]).items()
                  if k.split("garantia_")[1] in ("largo", "pct_mayusculas", "n_signos", "espacios_dobles", "exclamacion")})

    # permalink: el slug se arma con el título original; si no coincide con el actual, el título se editó
    m = re.match(r"^(https?)://([^/]+)/MLA\d+-(.*)-_JM$", fila["permalink"])
    f["perm_formato_estandar"] = int(m is not None)
    if m:
        f["perm_igual_titulo"] = int(m.group(3) == slug(titulo))
        f["perm_dif_largo"] = len(m.group(3)) - len(slug(titulo))
        f["perm_dominio"] = m.group(2)

    # precio
    for campo in ("price", "base_price"):
        p = fila[campo]
        entero = int(p)
        f[f"{campo}_es_entero"] = int(float(p).is_integer())
        f[f"{campo}_termina_9"] = int(entero % 10 == 9)
        f[f"{campo}_termina_99"] = int(entero % 100 == 99)
        f[f"{campo}_termina_0"] = int(entero % 10 == 0)
        f[f"{campo}_termina_00"] = int(entero % 100 == 0)
        f[f"{campo}_termina_000"] = int(entero % 1000 == 0)
        f[f"{campo}_digitos_significativos"] = len(str(entero).rstrip("0")) if entero else 0
    f["price_igual_base"] = int(fila["price"] == fila["base_price"])

    # fechas: segundos y milisegundos (Fase M), y diferencias cortas
    creada, actualizada = iso_a_ms(fila["date_created"]), iso_a_ms(fila["last_updated"])
    for nombre, ms in (("creada", creada), ("actualizada", actualizada), ("inicio", fila["start_time"]), ("fin", fila["stop_time"])):
        f[f"{nombre}_ms"] = ms % 1000
        f[f"{nombre}_segundo"] = (ms // 1000) % 60
    f["creada_menos_inicio_s"] = (creada - fila["start_time"]) / 1000
    f["actualizada_menos_creada_s"] = (actualizada - creada) / 1000

    # fotos
    fotos = fila["pictures"]
    ids = [p["id"] for p in fotos]
    if ids:
        nuevas = [PATRON_FOTO_NUEVA.match(i) for i in ids]
        viejas = [PATRON_FOTO_VIEJA.match(i) for i in ids]
        f["foto_pct_formato_mes"] = np.mean([m is not None and 1 <= int(m.group(3)) <= 12 and 2000 <= int(m.group(4)) <= 2016 for m in nuevas])
        f["foto_pct_mla_propio"] = np.mean([m is not None and m.group(2) == id_item for m in viejas])
        f["foto_ids_repetidos"] = len(ids) - len(set(ids))
        mes_creada = pd.Timestamp(fila["date_created"])
        meses = [(mes_creada.year - int(m.group(4))) * 12 + (mes_creada.month - int(m.group(3)))
                 for m in nuevas if m is not None and 1 <= int(m.group(3)) <= 12 and 2000 <= int(m.group(4)) <= 2016]
        f["foto_meses_antes_max"] = max(meses) if meses else np.nan
        f["foto_meses_antes_min"] = min(meses) if meses else np.nan
        prefijos = [int(m.group(1)) for m in viejas if m is not None]
        f["foto_prefijos_ordenados"] = int(prefijos == sorted(prefijos)) if len(prefijos) > 1 else np.nan
        f["foto_largo_prefijo"] = len(ids[0].split("-")[0])
        f["foto_largo_sufijo"] = len(ids[0].split("_")[-1])
        f["foto_pct_size_igual_max"] = np.mean([p.get("size") == p.get("max_size") for p in fotos])
        servidores = {re.sub(r"^https?://([^/]+)/.*$", r"\1", p.get("url", "")) for p in fotos}
        f["foto_n_servidores"] = len(servidores)
        f["foto_secure_akamai"] = np.mean(["akamai" in p.get("secure_url", "") for p in fotos])
        f["miniatura_es_primera"] = int(ids[0] in fila.get("thumbnail", ""))

    # orden de tags y de métodos de pago
    tags = fila["tags"]
    f["tags_ordenadas"] = int(tags == sorted(tags)) if len(tags) > 1 else np.nan
    f["tags_primera"] = tags[0] if tags else "(ninguna)"
    pagos = [p.get("id", "") for p in fila["non_mercado_pago_payment_methods"]]
    f["pagos_ordenados"] = int(pagos == sorted(pagos)) if len(pagos) > 1 else np.nan
    f["pagos_primero"] = pagos[0] if pagos else "(ninguno)"

    # descriptions: "{'id': 'MLA<item>-<número>'}"
    if fila["descriptions"]:
        m = re.search(r"MLA(\d+)-(\d+)", fila["descriptions"][0])
        f["desc_formato_estandar"] = int(m is not None)
        if m:
            f["desc_mla_propio"] = int(m.group(1) == id_item)
            f["desc_largo_numero"] = len(m.group(2))
            f["desc_numero"] = int(m.group(2))

    # variaciones y atributos
    variaciones = fila["variations"]
    if variaciones:
        f["var_pct_precio_decimal"] = np.mean([isinstance(v.get("price"), float) for v in variaciones])
        f["var_pct_custom_field"] = np.mean([v.get("seller_custom_field") is not None for v in variaciones])
        f["var_pct_con_fotos"] = np.mean([bool(v.get("picture_ids")) for v in variaciones])
    atributos = fila["attributes"]
    if atributos:
        f["attr_pct_value_id_vacio"] = np.mean([a.get("value_id") == "" for a in atributos])
        f["attr_pct_value_name_vacio"] = np.mean([a.get("value_name") == "" for a in atributos])
    f["parent_formato"] = estado(fila.get("parent_item_id")) + ("_MLA" if str(fila.get("parent_item_id", "")).startswith("MLA") else "")
    return f


inicio = time.time()
formato_train = pd.DataFrame([{**genericas(f), **especificas(f)} for f in filas_train])
formato_test = pd.DataFrame([{**genericas(f), **especificas(f)} for f in filas_test]).reindex(columns=formato_train.columns)
# Se descartan las constantes (un solo valor en train + test)
constantes = [c for c in formato_train if pd.concat([formato_train[c], formato_test[c]]).nunique(dropna=False) <= 1]
formato_train, formato_test = formato_train.drop(columns=constantes), formato_test.drop(columns=constantes)
print(f"{formato_train.shape[1]} features de formato ({len(constantes)} constantes descartadas) en {(time.time() - inicio) / 60:.1f} min")

# Campo de origen (para agrupar por campo y mostrar ejemplos crudos)
def origen(columna):
    if "|" in columna:
        return columna.split("|")[1].split(".")[0].split("[")[0]
    if columna.startswith("base_price_") or columna.endswith("|base_price"):
        return "price"                            # price y base_price coinciden en casi todas las filas: un solo campo
    prefijos = {"titulo": "title", "garantia": "warranty", "perm": "permalink", "price": "price",
                "creada": "date_created", "actualizada": "last_updated", "inicio": "start_time", "fin": "stop_time",
                "foto": "pictures", "miniatura": "thumbnail", "tags": "tags", "pagos": "non_mercado_pago_payment_methods",
                "desc": "descriptions", "var": "variations", "attr": "attributes", "parent": "parent_item_id"}
    return prefijos.get(columna.split("_")[0], columna)


YA_PROBADO = {"date_created": "Fase M", "last_updated": "Fase M", "start_time": "Fase M", "stop_time": "Fase M (duración)",
              "thumbnail": "Fase G", "secure_thumbnail": "Fase G", "price": "07 (redondez, al límite del ruido)"}

224 features de formato (46 constantes descartadas) en 0.3 min


## 2. Ranking dentro de la zona de duda (4 folds → quinto)

**Qué hace `univariada(x)`:** para cada fold k:
1. Con las filas de la zona de los **otros 4 folds**, agrupa la feature:
   - si tiene ≤ 20 valores, cada valor es un grupo;
   - si no, 10 cuantiles;
   - NaN es un grupo propio;
   - los grupos con menos de 30 filas usan el % `new` general de la zona.
2. Cada grupo predice la clase mayoritaria de esas filas.
3. Evalúa en las filas de la zona del fold k.

**Qué sale:**
- `acc_sola`: accuracy en la zona;
- `ganancia`: `acc_sola` − accuracy de predecir siempre la clase mayoritaria (lo que separa la feature);
- `folds_ganancia`: en cuántos folds la ganancia es > 0;
- `dif_pct_new`: % `new` del grupo más `new` menos el del más `used`. Los dos grupos se eligen con los 4 folds (≥ 30 filas) y se miden en el quinto (≥ 10 filas).

In [3]:
def a_texto(x):
    # Ojo pandas 3: .astype(str) deja los NaN como faltantes (no como "nan"), y groupby los descarta
    return pd.Series(x).astype(str).fillna("nan").to_numpy()


def grupos_de(x, entrena):
    x = pd.Series(x)
    if x.dtype == object or x.dtype == "str" or x.nunique(dropna=True) <= 20:
        return a_texto(x)
    cortes = np.unique(np.nanquantile(x[entrena].astype(float), np.linspace(0, 1, 11)))
    if len(cortes) < 2:
        return a_texto(x)
    g = np.digitize(x.astype(float), cortes[1:-1]).astype(str)
    return np.where(x.isna(), "nan", g)


def univariada(x):
    acc, base, dif = [], [], []
    for k in range(1, 6):
        entrena, evalua = zona & (folds != k), zona & (folds == k)
        g = grupos_de(x, entrena)
        resumen = pd.DataFrame({"g": g[entrena], "y": y_np[entrena]}).groupby("g")["y"].agg(["mean", "size"])
        previa = y_np[entrena].mean()
        prob_grupo = resumen["mean"].where(resumen["size"] >= 30, previa)
        p = pd.Series(g[evalua]).map(prob_grupo).fillna(previa).to_numpy()
        acc.append(((p >= 0.5) == y_np[evalua]).mean())
        base.append(((previa >= 0.5) == y_np[evalua]).mean())
        grandes = resumen[resumen["size"] >= 30]
        if len(grandes) >= 2:
            g_ev, y_ev = g[evalua], y_np[evalua]
            alto, bajo = grandes["mean"].idxmax(), grandes["mean"].idxmin()
            if (g_ev == alto).sum() >= 10 and (g_ev == bajo).sum() >= 10:
                dif.append(y_ev[g_ev == alto].mean() - y_ev[g_ev == bajo].mean())
    ganancia = np.array(acc) - np.array(base)
    return {"acc_sola": np.mean(acc), "ganancia": ganancia.mean(), "folds_ganancia": int((np.round(ganancia, 10) > 0).sum()),
            "dif_pct_new": np.mean(dif) if dif else np.nan}


inicio = time.time()
ranking = pd.DataFrame({c: univariada(formato_train[c]) for c in formato_train}).T
ranking["campo"] = [origen(c) for c in ranking.index]
ranking["ya_probado"] = ranking["campo"].map(YA_PROBADO).fillna("")
ranking["cobertura_zona"] = [formato_train.loc[zona, c].notna().mean() for c in ranking.index]
ranking = ranking.sort_values(["ganancia", "dif_pct_new"], ascending=False)
ranking.to_csv(os.path.join(carpeta_lab, "artefactos_ranking.csv"))
print(f"Ranking de {len(ranking)} features en {(time.time() - inicio) / 60:.1f} min")
print("Ganancia > 0 en 5/5 folds:", int((ranking["folds_ganancia"] == 5).sum()), "| en ≥ 4/5:", int((ranking["folds_ganancia"] >= 4).sum()))
ranking.head(30).round(4)

/Users/nicolasscravaglieri/Downloads/kaggle-ap-2026/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1573: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


Ranking de 224 features en 0.1 min
Ganancia > 0 en 5/5 folds: 46 | en ≥ 4/5: 62


,acc_sola,ganancia,folds_ganancia,dif_pct_new,campo,ya_probado,cobertura_zona
foto_largo_prefijo,0.5332,0.0384,5.0,0.0304,pictures,,0.9876
base_price_termina_00,0.5307,0.0359,5.0,0.0661,price,"07 (redondez, al límite del ruido)",1.0000
price_termina_00,0.5306,0.0357,5.0,0.0659,price,"07 (redondez, al límite del ruido)",1.0000
largo|pictures[0].id,0.5282,0.0334,5.0,-0.0571,pictures,,0.9876
largo|pictures[0].secure_url,0.5282,0.0334,5.0,-0.0571,pictures,,0.9876
largo|pictures[0].url,0.5282,0.0334,5.0,-0.0571,pictures,,0.9876
largo|secure_thumbnail,0.5282,0.0334,5.0,-0.0571,secure_thumbnail,Fase G,0.9876
largo|thumbnail,0.5282,0.0334,5.0,-0.0571,thumbnail,Fase G,0.9876
foto_meses_antes_max,0.5234,0.0285,5.0,0.0907,pictures,,0.9305
base_price_digitos_significativos,0.5207,0.0259,5.0,0.1318,price,"07 (redondez, al límite del ruido)",1.0000


**Filtro antes de elegir** (agregado después de mirar el primer ranking). Algunas features del barrido no son artefactos nuevos:
- `largo|listing_type_id` es el **valor** disfrazado ("free" tiene 4 letras y "gold_special" 12);
- `estado|parent_item_id` es `tiene_parent_item`, y `n|pictures` es `n_fotos`;
- `price_termina_00` y `base_price_termina_00` son casi la misma columna.

**Qué hace `casi_funcion(a, b)`:** agrupa por `a` y mira qué % de filas tiene en `b` el valor más común de su grupo. Si es ≥ 99,5%, `b` es (casi) una función de `a`: no agrega nada que `a` no diga.

**Regla para elegir:** se recorre el ranking de mayor a menor ganancia, solo con features que ganan en 5/5 folds, no probadas en las Fases G ni M. Se saltea una candidata si:
- es casi función de una columna que **el modelo ya usa** (categóricas, flags y conteos de `features.py`), o
- es casi función de una ya elegida, o al revés (duplicado).

Se eligen hasta 5. No se mira el target para filtrar: solo las features.

In [4]:
def casi_funcion(a, b, umbral=0.995):
    # ¿b es (casi) una función de a? % de filas cuyo b es el más común dentro de su grupo de a
    a, b = a_texto(a), a_texto(b)
    tabla = pd.DataFrame({"a": a, "b": b}).value_counts(dropna=False).reset_index(name="cant")
    assert tabla["cant"].sum() == len(a)
    return tabla.groupby("a")["cant"].max().sum() / len(a) >= umbral


COLUMNAS_DEL_MODELO = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state", "currency_id", "status",
                       "n_fotos", "n_atributos", "n_variaciones", "n_metodos_pago", "tiene_video", "tiene_garantia",
                       "tiene_descuento", "tiene_oferta", "tiene_parent_item", "es_tienda_oficial", "free_shipping",
                       "local_pick_up", "automatic_relist", "accepts_mercadopago", "c_tags", "c_pagos", "c_attr"]
grupos_modelo = {m: a_texto(train[m]) for m in COLUMNAS_DEL_MODELO}

ELEGIDAS, descartes = [], {}
for c in ranking.index:
    fila = ranking.loc[c]
    if len(ELEGIDAS) == 5:
        break
    if fila["folds_ganancia"] < 5 or fila["ya_probado"].startswith("Fase"):
        continue
    g = grupos_de(formato_train[c], zona)
    motivo = next((f"función de {m} (modelo)" for m, gm in grupos_modelo.items() if casi_funcion(gm, g)), None)
    if motivo is None:
        for e in ELEGIDAS:
            ge = grupos_de(formato_train[e], zona)
            if casi_funcion(ge, g) or casi_funcion(g, ge):
                motivo = f"duplica a {e}"
                break
    if motivo:
        descartes[c] = motivo
    else:
        ELEGIDAS.append(c)
print("Descartadas por el filtro:")
for c, motivo in descartes.items():
    print(f"  {c:45s} {motivo}")
ranking.loc[ELEGIDAS].round(4)

Descartadas por el filtro:
  price_termina_00                              duplica a base_price_termina_00
  largo|pictures[0].secure_url                  duplica a largo|pictures[0].id
  largo|pictures[0].url                         duplica a largo|pictures[0].id


,acc_sola,ganancia,folds_ganancia,dif_pct_new,campo,ya_probado,cobertura_zona
foto_largo_prefijo,0.5332,0.0384,5.0,0.0304,pictures,,0.9876
base_price_termina_00,0.5307,0.0359,5.0,0.0661,price,"07 (redondez, al límite del ruido)",1.0000
largo|pictures[0].id,0.5282,0.0334,5.0,-0.0571,pictures,,0.9876
foto_meses_antes_max,0.5234,0.0285,5.0,0.0907,pictures,,0.9305
base_price_digitos_significativos,0.5207,0.0259,5.0,0.1318,price,"07 (redondez, al límite del ruido)",1.0000


## 3. Elegidas: ¿aparecen igual en el test? Ejemplos crudos

Las elegidas por la regla de arriba. Para cada una:
- frecuencia de cada grupo en train y en test (si un artefacto no aparece en el test, no sirve);
- % `new` por grupo en todo el train y en la zona;
- filas crudas de ejemplo de la zona: las de los grupos más `new` y más `used`.

In [5]:
print("Elegidas:", ELEGIDAS)

for c in ELEGIDAS:
    g_tr = grupos_de(formato_train[c], zona)
    g_te = grupos_de(pd.concat([formato_train[c], formato_test[c]], ignore_index=True), np.r_[zona, np.zeros(len(formato_test), bool)])[n:]
    tabla = pd.DataFrame({
        "% filas train": pd.Series(g_tr).value_counts(normalize=True),
        "% filas test": pd.Series(g_te).value_counts(normalize=True),
        "% new train": pd.Series(y_np).groupby(g_tr).mean(),
        "filas zona": pd.Series(g_tr[zona]).value_counts(),
        "% new zona": pd.Series(y_np[zona]).groupby(g_tr[zona]).mean(),
    }).sort_values("% filas train", ascending=False).head(12)
    print(f"\n=== {c} (campo {origen(c)}) ===")
    display(tabla.round(3))

    resumen = tabla[tabla["filas zona"] >= 30]
    for etiqueta, grupo in (("más new", resumen["% new zona"].idxmax()), ("más used", resumen["% new zona"].idxmin())):
        idx = np.where(zona & (g_tr == grupo))[0][:3]
        ejemplos = pd.DataFrame({
            "condition": crudo["condition"].iloc[idx].to_numpy(), "P(new) entrega": ref_final[idx].round(3),
            "feature": formato_train[c].iloc[idx].to_numpy(), "title": crudo["title"].iloc[idx].to_numpy(),
            "crudo": [json.dumps(filas_train[i].get(origen(c)), ensure_ascii=False)[:150] for i in idx]}, index=idx)
        print(f"-- grupo {etiqueta}: {grupo}")
        display(ejemplos)

Elegidas: ['foto_largo_prefijo', 'base_price_termina_00', 'largo|pictures[0].id', 'foto_meses_antes_max', 'base_price_digitos_significativos']

=== foto_largo_prefijo (campo pictures) ===


,% filas train,% filas test,% new train,filas zona,% new zona
6.0,0.513,0.510,0.573,4226.0,0.526
5.0,0.310,0.309,0.538,2848.0,0.453
4.0,0.165,0.167,0.437,1300.0,0.525
nan,0.008,0.008,0.320,106.0,0.462
3.0,0.004,0.004,0.463,35.0,0.600
2.0,0.001,0.001,0.449,12.0,0.500
1.0,0.000,0.000,0.375,NaN,NaN


-- grupo más new: 3.0


,condition,P(new) entrega,feature,title,crudo
3624,new,0.631,3.0,Star Wars The Legacy Collection Senator Padmé Amidala,"[{""size"": ""500x488"", ""secure_url"": ""https://a248.e.akamai.net/mla-s2-p.mlstatic.com/118-MLA1647652056_072013-O.jpg"",..."
3903,new,0.153,3.0,Xlr Mic Combiner Pasivo,"[{""size"": ""269x187"", ""secure_url"": ""https://a248.e.akamai.net/mla-s1-p.mlstatic.com/876-MLA4083917738_072013-O.jpg"",..."
4819,used,0.444,3.0,Chata Borde Bajo (escala H0),"[{""size"": ""320x240"", ""secure_url"": ""https://a248.e.akamai.net/mla-s1-p.mlstatic.com/868-MLA1284872596_072013-O.jpg"",..."


-- grupo más used: 5.0


,condition,P(new) entrega,feature,title,crudo
11,new,0.178,5.0,El Dueño | Luis Majul | Editorial Planeta,"[{""size"": ""500x439"", ""secure_url"": ""https://a248.e.akamai.net/mla-s2-p.mlstatic.com/17951-MLA8582354300_082014-O.jpg..."
22,used,0.237,5.0,Mineral Roca Cristal Microclino Topacios Biterminados Cuarzo,"[{""size"": ""500x375"", ""secure_url"": ""https://a248.e.akamai.net/mla-s1-p.mlstatic.com/17825-MLA6757949832_082014-O.jpg..."
23,new,0.787,5.0,Pollera Thai,"[{""size"": ""375x500"", ""secure_url"": ""https://a248.e.akamai.net/mla-s1-p.mlstatic.com/17488-MLA3260463496_082014-O.jpg..."


... (20 outputs left out)

-- grupo más new: 4


,condition,P(new) entrega,feature,title,crudo
177,new,0.046,4,Coche Lite Ryder H 272q Infanti,2115.0
322,new,0.688,4,Social Distortion Somewhere Between Heaven & Hell Lp 2011,1199.0
1135,new,0.760,4,Lumia 535 Libre De Fabrica Nuevo Vendo Sony Lg Samsung Nokia,3399.0


-- grupo más used: 1


,condition,P(new) entrega,feature,title,crudo
197,new,0.111,1,Tapas 382,20.0
218,used,0.425,1,Durometro Portatil Tipo Leeb,10000.0
220,new,0.722,1,Campera Columbia Sportswear Para Chica.,1000.0


## 4. Sumadas al HistGB y al promedio final (comparación pareada contra la entrega)

**Qué hace `armar(modelo, extras)`:** el pipeline de la entrega (`armar_pipeline_histgb_combos()` / `armar_pipeline_final()`) con dos cambios, sin tocar `features.py`:
1. El transformer `texto` se reemplaza por `passthrough` de la columna exacta de la variante a (`columna_a(k)`, misma posición). Igual que en la 31.
2. Se agrega al **final** del `ColumnTransformer` un `passthrough` con las features nuevas. En el CatBoost quedan después de las categóricas, así que `INDICES_CAT` no cambia.

Las features de texto (como `perm_dominio` o `estado|...`) se pasan a números con un código fijo, sacado de train + test (no mira el target). NaN queda como NaN: HistGB y CatBoost lo manejan.

**Control:** el HistGB sin extras tiene que dar exactamente las OOF guardadas de la entrega.

**Candidatas:** cada elegida sola y todas juntas, en HistGB y CatBoost. Promedio final = 50/50.

In [6]:
def a_numero(c):
    todo = pd.concat([formato_train[c], formato_test[c]], ignore_index=True)
    if todo.dtype == object:
        codigos = {v: i for i, v in enumerate(sorted(todo.dropna().astype(str).unique()))}
        return formato_train[c].map(lambda v: codigos.get(str(v), np.nan) if pd.notna(v) else np.nan).astype(float)
    return formato_train[c].astype(float)


nombres_extra = {c: f"fmt_{i}" for i, c in enumerate(formato_train.columns)}
train_fmt = pd.concat([train, pd.DataFrame({nombres_extra[c]: a_numero(c) for c in ELEGIDAS})], axis=1)


def armar(modelo, extras):
    pipeline = armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", "passthrough", ["t_texto"])
    if extras:
        transformers.append(("formato", "passthrough", [nombres_extra[c] for c in extras]))
    return pipeline


def correr_fold(k, modelo, extras):
    entrena, valida = folds != k, folds == k
    X = train_fmt.assign(t_texto=columna_a(k))
    params = {"modelo__cat_features": INDICES_CAT} if modelo == "catboost" else {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar(modelo, extras).fit(X[entrena], y[entrena], **params)
    return k, pipeline.predict_proba(X[valida])[:, 1]


def oof(modelo, extras):
    prob = np.empty(n)
    for k, p in Parallel(n_jobs=5)(delayed(correr_fold)(k, modelo, extras) for k in range(1, 6)):
        prob[folds == k] = p
    return prob


# Control + medición de tiempos (antes de correr todo)
inicio = time.time()
control = oof("histgb", [])
print(f"Control HistGB sin extras: diferencia máxima con la entrega {np.abs(control - ref_hgb).max():.2e} "
      f"({(time.time() - inicio) / 60:.1f} min, 5 folds en paralelo)")
inicio = time.time()
correr_fold(1, "catboost", ELEGIDAS)
print(f"CatBoost, 1 fold con las 5 extras: {(time.time() - inicio) / 60:.1f} min")

Control HistGB sin extras: diferencia máxima con la entrega 1.11e-16 (0.5 min, 5 folds en paralelo)


CatBoost, 1 fold con las 5 extras: 0.4 min


In [7]:
CANDIDATAS = {c: [c] for c in ELEGIDAS}
if len(ELEGIDAS) > 1:
    CANDIDATAS["todas juntas: " + " + ".join(ELEGIDAS)] = ELEGIDAS   # el nombre lleva la lista: la caché no se confunde

archivo_oof = os.path.join(carpeta_lab, "oof_artefactos.csv")
guardado = pd.read_csv(archivo_oof) if os.path.exists(archivo_oof) else pd.DataFrame({"fold": folds, "y": y_np})
if os.environ.get("SOLO_MEDIR"):
    CANDIDATAS = {}
for nombre, extras in CANDIDATAS.items():
    for modelo in ["histgb", "catboost"]:
        col = f"{modelo} | {nombre}"
        if col not in guardado:
            inicio = time.time()
            guardado[col] = oof(modelo, extras)
            guardado.to_csv(archivo_oof, index=False)
            print(f"{col:70s} {(time.time() - inicio) / 60:.1f} min", flush=True)

histgb | todas juntas: foto_largo_prefijo + base_price_termina_00 + largo|pictures[0].id + foto_meses_antes_max + base_price_digitos_significativos 0.5 min


catboost | todas juntas: foto_largo_prefijo + base_price_termina_00 + largo|pictures[0].id + foto_meses_antes_max + base_price_digitos_significativos 1.6 min


In [8]:
filas, por_fold = {}, pd.DataFrame({"entrega": acc_por_fold(ref_final)}, index=pd.Index(range(1, 6), name="fold"))
for nombre in CANDIDATAS:
    p_h = guardado[f"histgb | {nombre}"].to_numpy()
    p_f = 0.5 * guardado[f"catboost | {nombre}"].to_numpy() + 0.5 * p_h
    vs_h, vs_f = comparar(acc_por_fold(p_h), acc_por_fold(ref_hgb)), comparar(acc_por_fold(p_f), acc_por_fold(ref_final))
    filas[nombre] = {"histgb": vs_h["media"], "histgb_dif": vs_h["dif_media"], "histgb_folds": vs_h["folds_gana"], "histgb_pasa": vs_h["pasa"],
                     "final": vs_f["media"], "final_dif": vs_f["dif_media"], "final_desvio": vs_f["dif_desvio"],
                     "final_folds": vs_f["folds_gana"], "final_pasa": vs_f["pasa"]}
    por_fold[f"dif | {nombre}"] = acc_por_fold(p_f) - acc_por_fold(ref_final)
print(f"Entrega: HistGB {acc_por_fold(ref_hgb).mean():.4f} | final {acc_por_fold(ref_final).mean():.4f}")
tabla = pd.DataFrame(filas).T
tabla.to_csv(os.path.join(carpeta_lab, "artefactos.csv"))
display(tabla.round(5))
por_fold.loc["media"] = por_fold.mean()
por_fold.round(5)

Entrega: HistGB 0.9425 | final 0.9435


,histgb,histgb_dif,histgb_folds,histgb_pasa,final,final_dif,final_desvio,final_folds,final_pasa
foto_largo_prefijo,0.942657,0.000143,3,False,0.943814,0.000286,0.000487,3,False
base_price_termina_00,0.9425,-0.000014,3,False,0.943629,0.0001,0.000787,2,False
largo|pictures[0].id,0.942786,0.000271,4,False,0.943586,0.000057,0.00042,1,False
foto_meses_antes_max,0.9425,-0.000014,3,False,0.943743,0.000214,0.000278,4,False
base_price_digitos_significativos,0.942743,0.000229,2,False,0.943543,0.000014,0.000514,2,False
todas juntas: foto_largo_prefijo + base_price_termina_00 + largo|pictures[0].id + foto_meses_antes_max + base_price_digitos_significativos,0.942929,0.000414,4,False,0.943757,0.000229,0.00066,2,False


,entrega,dif | foto_largo_prefijo,dif | base_price_termina_00,dif | largo|pictures[0].id,dif | foto_meses_antes_max,dif | base_price_digitos_significativos,dif | todas juntas: foto_largo_prefijo + base_price_termina_00 + largo|pictures[0].id + foto_meses_antes_max + base_price_digitos_significativos
fold,,,,,,,
1,0.94321,0.00064,0.00129,0.00086,0.00043,0.00093,0.00107
2,0.94221,-0.00007,-0.00007,-0.00029,0.00050,-0.00021,-0.00029
3,0.94421,0.00036,-0.00114,-0.00029,0.00014,-0.00043,-0.00036
4,0.94350,0.00093,0.00043,0.00000,-0.00029,-0.00043,0.00100
5,0.94450,-0.00043,0.00000,0.00000,0.00029,0.00021,-0.00029
media,0.94353,0.00029,0.00010,0.00006,0.00021,0.00001,0.00023
